# Exploración Profunda: API Data360 del Banco Mundial

Este cuaderno profundiza en la estructura de la API **Data360**, analizando los metadatos disponibles: cobertura temporal, áreas geográficas y categorías de datos (temas).

**API Endpoint**: `https://data360api.worldbank.org`  
**Documentación**: Data360 Open API Spec

In [ ]:
import requests
import pd = pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Configuración base
BASE_URL = "https://data360api.worldbank.org/data360"
sns.set_theme(style="whitegrid")

## 1. Cobertura Temporal

¿Qué años de información podemos extraer? Analizaremos el rango temporal disponible para un indicador global representativo.

In [ ]:
def get_time_range(indicator_id="WB_WDI_SP_POP_TOTL", area="WLD"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "REF_AREA": area}
    resp = requests.get(url, params=params).json()
    years = [int(v['TIME_PERIOD']) for v in resp['value'] if v['TIME_PERIOD'].isdigit()]
    return min(years), max(years), len(years)

min_y, max_y, total_y = get_time_range()
print(f"Rango temporal disponible: {min_y} - {max_y} ({total_y} años)")

## 2. Cobertura Geográfica (Países y Áreas)

La API no solo incluye países, sino también agregados regionales (ej. América Latina, Europa) y niveles de ingreso (ej. Países de ingreso bajo).

In [ ]:
def get_available_areas(indicator_id="WB_WDI_SP_POP_TOTL"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "isLatestData": True}
    resp = requests.get(url, params=params).json()
    areas = [v['REF_AREA'] for v in resp['value']]
    return sorted(list(set(areas)))

areas = get_available_areas()
print(f"Total de áreas/países disponibles: {len(areas)}")
print("Muestra de códigos de área (ISO3):", areas[:25])

## 3. Categorías de Datos (Temas)

Los indicadores están agrupados por temas. Utilizaremos el endpoint de metadatos para descubrir las categorías principales.

In [ ]:
def get_topics(top=200):
    url = f"{BASE_URL}/metadata"
    query = {"query": f"&$filter=series_description/database_id eq 'WB_WDI'&$select=series_description/topics/name&$top={top}"}
    resp = requests.post(url, json=query).json()
    
    unique_topics = set()
    if 'value' in resp:
        for item in resp['value']:
            sd = item.get('series_description', {})
            # La API puede devolver series_description como dict o list segun el dataset
            entries = sd if isinstance(sd, list) else [sd]
            for entry in entries:
                for t in entry.get('topics', []):
                    if t.get('name'): unique_topics.add(t['name'])
    return sorted(list(unique_topics))

topics = get_topics()
print(f"Temas principales detectados ({len(topics)}):")
for t in topics: print(f"- {t}")

## 4. Visualización Exploratoria

### Crecimiento Comparativo en el Cono Sur
Utilizando los datos descubiertos, comparamos el crecimiento poblacional relativo entre países vecinos.

In [ ]:
def get_indicator_data(indicator, countries):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator, "REF_AREA": ",".join(countries)}
    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp['value'])
    df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'])
    df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'])
    return df

df_cono_sur = get_indicator_data("WB_WDI_SP_POP_TOTL", ["ARG", "CHL", "URY", "BRA"])

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_cono_sur, x='TIME_PERIOD', y='OBS_VALUE', hue='REF_AREA')
plt.title('Población Total en el Cono Sur (1960-2023)', fontsize=15)
plt.ylabel('Población')
plt.yscale('log') # Escala logarítmica para ver mejor las tendencias
plt.show()

## Resumen Técnico de la Exploración

- **Rango Temporal**: 1960 a 2024. Ideal para series de largo plazo.
- **Cobertura Espacial**: 265 códigos de área, permitiendo análisis país por país o por bloques económicos (G20, OCDE, etc.).
- **Temáticas**: Diversidad desde agricultura y medio ambiente hasta políticas económicas y agua.
- **Facilidad de Uso**: La API permite filtrado complejo mediante OData (usado en el endpoint de metadatos).